<a href="https://colab.research.google.com/github/AISTAC/dev-environment-exercise/blob/main/Chatbot_with_Azure_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Traditional Chatbot
**Project 1 — Rule-based chatbot (Google Colab version)**

This notebook implements a simple, traditional (rule-based / keyword-matching) chatbot.
No machine learning framework is required — this satisfies the assignment's requirement that the bot support simple interactions, list its capabilities, handle malformed input, and be structured so it could later be extended to call an AI-as-a-service API.

## 1. Chatbot implementation

In [ ]:
# @title
import re
import random
from datetime import datetime

BOT_NAME = "CumberlandBot"

RULES = [
    (r"\b(hi|hello|hey|greetings)\b",
     ["Hello! I'm {name}. Type 'help' to see what I can do.",
      "Hi there! How can I help you today?"]),

    (r"\b(are you god.s favorite|god.s favorite child)\b",
     ["I am god's favorite child."]),

    (r"\b(help|capabilities|what can you do)\b",
     ["I can: greet you, tell you the time, do simple math (e.g. 'add 2 and 3'), "
      "answer basic questions about myself, and say goodbye. "
      "Type 'help' anytime to see this list again."]),

    (r"\bwhat('?s| is) your name\b",
     ["My name is {name}, a simple rule-based chatbot."]),

    (r"\b(time|what time is it)\b",
     [lambda: f"The current time is {datetime.now().strftime('%H:%M:%S')}."]),

    (r"\b(date|what.s the date|today.s date)\b",
     [lambda: f"Today's date is {datetime.now().strftime('%Y-%m-%d')}."]),

    (r"\bhow are you\b",
     ["I'm just a program, but I'm running smoothly. Thanks for asking!"]),

    (r"\b(bye|goodbye|exit|quit)\b",
     ["Goodbye! Thanks for chatting.", "See you later!"]),
]


def normalize(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def generate_response(user_input: str) -> str:
    if user_input is None:
        return "I didn't receive any input. Could you try typing something?"

    text = normalize(str(user_input))

    if text == "":
        return "It looks like you didn't type anything. Try 'help' to see what I can do."

    if not re.search(r"[a-z0-9]", text):
        return "I couldn't understand that — it looks like it has no readable words. Try again?"

    add_match = re.search(r"\badd (\d+) and (\d+)\b", text)
    if add_match:
        a, b = int(add_match.group(1)), int(add_match.group(2))
        return f"{a} + {b} = {a + b}"

    for pattern, responses in RULES:
        if re.search(pattern, text):
            choice = random.choice(responses)
            if callable(choice):
                return choice()
            return choice.format(name=BOT_NAME)

    return ("I'm not sure how to respond to that yet. "
            "Type 'help' to see what I can do, or try rephrasing.")



## 2. Automated test cases (screenshot this cell's output for your report)

In [ ]:
test_inputs = [
    "hello",
    "help",
    "what is your name",
    "add 4 and 5",
    "time",
    "",
    "!!!???",
    "asdkjfh gibberish",
    "bye",
]

for text in test_inputs:
    print(f"You: {text!r}")
    print(f"{BOT_NAME}: {generate_response(text)}\n")

You: 'hello'
CumberlandBot: Hello! I'm CumberlandBot. Type 'help' to see what I can do.

You: 'help'
CumberlandBot: I can: greet you, tell you the time, do simple math (e.g. 'add 2 and 3'), answer basic questions about myself, and say goodbye. Type 'help' anytime to see this list again.

You: 'what is your name'
CumberlandBot: My name is CumberlandBot, a simple rule-based chatbot.

You: 'add 4 and 5'
CumberlandBot: 4 + 5 = 9

You: 'time'
CumberlandBot: The current time is 04:51:16.

You: ''
CumberlandBot: It looks like you didn't type anything. Try 'help' to see what I can do.

You: '!!!???'
CumberlandBot: I couldn't understand that — it looks like it has no readable words. Try again?

You: 'asdkjfh gibberish'
CumberlandBot: I'm not sure how to respond to that yet. Type 'help' to see what I can do, or try rephrasing.

You: 'bye'
CumberlandBot: See you later!



## 3. Interactive chat
Run this cell, then type in the input box that appears below it. Type `bye`, `exit`, or `quit` to stop.

> Note: this uses Python's built-in `input()`, which Colab supports natively — no extra setup needed.

In [ ]:
print(f"{BOT_NAME}: Hello! Type 'help' for a list of things I can do, or 'bye' to exit.")
while True:
    user_input = input("You: ")
    response = generate_response(user_input)
    print(f"{BOT_NAME}: {response}")
    if re.search(r"\b(bye|goodbye|exit|quit)\b", normalize(user_input)):
        break

CumberlandBot: Hello! Type 'help' for a list of things I can do, or 'bye' to exit.
You: bye
CumberlandBot: See you later!


## 4. Connect to Azure AI Language (Sentiment Analysis)This part adds a real cloud AI service to our chatbot: Azure AI Language's **sentiment analysis**. Sentiment analysis reads a sentence and tells us if it sounds positive, negative, neutral, or mixed.**Before running this section**, make sure you have:1. Created an Azure AI Language resource in the Azure Portal, and2. Copied your **Key** and **Endpoint** into Colab's Secrets (the key icon 🔑 on the left sidebar), saved under the exact names `AZURE_LANGUAGE_KEY` and `AZURE_LANGUAGE_ENDPOINT`, with **Notebook access** turned on for both.See the separate step-by-step guide for how to do both of those things.

## 5. Combine the Rule-Based Chatbot with Azure AIRight now, when our chatbot does not recognize what you typed, it gives a generic "I'm not sure" message. We are going to upgrade that fallback: instead of a generic message, the chatbot will now ask Azure AI what sentiment your message has, and reply with that instead. This is exactly the kind of extension our chatbot was designed for from the start.

## 6. Test the AI-Connected ChatbotScreenshot this cell's output for your report — it shows the chatbot calling Azure AI for messages it doesn't have a rule for.

## 7. Interactive Chat (AI-Connected Version)Same as before, but now unrecognized messages get analyzed by Azure AI instead of a generic reply. Type `bye`, `exit`, or `quit` to stop.

In [15]:
# Part 4: Connect to Azure AI Language (Sentiment Analysis)
!pip install --quiet azure-ai-textanalytics

from google.colab import userdata
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

AZURE_LANGUAGE_KEY = userdata.get('AZURE_LANGUAGE_KEY')
AZURE_LANGUAGE_ENDPOINT = userdata.get('AZURE_LANGUAGE_ENDPOINT')

azure_client = TextAnalyticsClient(
    endpoint=AZURE_LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_LANGUAGE_KEY)
)

print('Azure client created successfully.')

def analyze_sentiment(text):
    """Send text to Azure AI Language and return (sentiment_label, confidence)."""
    result = azure_client.analyze_sentiment(documents=[text])[0]
    scores = result.confidence_scores
    if result.sentiment == "positive":
        confidence = scores.positive
    elif result.sentiment == "negative":
        confidence = scores.negative
    elif result.sentiment == "neutral":
        confidence = scores.neutral
    else:
        confidence = max(scores.positive, scores.neutral, scores.negative)
    return result.sentiment, confidence

Azure client created successfully.


In [16]:
# Part 5: Combine the Rule-Based Chatbot with Azure AI
FALLBACK_TEXT = ("I'm not sure how to respond to that yet. "
                  "Type 'help' to see what I can do, or try rephrasing.")

def generate_response_with_ai(user_input):
    base_response = generate_response(user_input)
    if base_response == FALLBACK_TEXT:
        sentiment, confidence = analyze_sentiment(user_input)
        return (f"I don't have a specific rule for that, but Azure AI Language thinks your "
                f"message sounds {sentiment} (confidence: {confidence:.0%}).")
    return base_response

In [17]:
# Part 6: Test the AI-Connected Chatbot
ai_test_inputs = [
    "hello",
    "I am so excited about this project!",
    "This is the worst chatbot I have ever used.",
    "The sky is blue today.",
]

for text in ai_test_inputs:
    print(f"You: {text!r}")
    print(f"{BOT_NAME}: {generate_response_with_ai(text)}\n")

You: 'hello'
CumberlandBot: Hi there! How can I help you today?

You: 'I am so excited about this project!'
CumberlandBot: I don't have a specific rule for that, but Azure AI Language thinks your message sounds positive (confidence: 100%).

You: 'This is the worst chatbot I have ever used.'
CumberlandBot: I don't have a specific rule for that, but Azure AI Language thinks your message sounds negative (confidence: 100%).

You: 'The sky is blue today.'
CumberlandBot: I don't have a specific rule for that, but Azure AI Language thinks your message sounds neutral (confidence: 99%).



In [18]:
# Part 7: Interactive Chat (AI-Connected Version)
print(f"{BOT_NAME}: Hello! Type 'help' for a list of things I can do, or 'bye' to exit.")
while True:
    user_input = input("You: ")
    response = generate_response_with_ai(user_input)
    print(f"{BOT_NAME}: {response}")
    if re.search(r"\b(bye|goodbye|exit|quit)\b", normalize(user_input)):
        break

CumberlandBot: Hello! Type 'help' for a list of things I can do, or 'bye' to exit.
You: CAN YOU TELL ME WHAT 1000 x23
CumberlandBot: I don't have a specific rule for that, but Azure AI Language thinks your message sounds neutral (confidence: 99%).
You: BYE
CumberlandBot: Goodbye! Thanks for chatting.
